# Diabetes Readmission Prediction Using Structured Health Data

**CS 549 Machine Learning | San Diego State University | Spring 2026**

**Group 20:** Niki Liao, Javi Garcia, Adrian Serrano, Jackson Adams

This project predicts whether a diabetic patient will be readmitted to the hospital within 30 days of discharge. We use the Diabetes 130-US Hospitals dataset (1999-2008) from the UCI Machine Learning Repository and compare four classification models: Logistic Regression, Random Forest, K-Nearest Neighbors, and XGBoost.

## 1. Setup and Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time
import warnings
import os

from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report,
    roc_curve, auc, precision_recall_curve
)
from sklearn.calibration import calibration_curve

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

print('All packages loaded successfully')

: 

## 2. Data Loading and Exploration

The dataset contains over 100,000 inpatient records collected across 130 U.S. hospitals between 1999 and 2008. 
Each record includes around 50 features covering patient demographics, admission details, diagnoses (ICD-9 codes), 
medications prescribed, and lab test results.

We load the data with `?` treated as NaN values since thats how missing data is encoded in this dataset.

In [ ]:
df = pd.read_csv('data/diabetic_data.csv', na_values='?', low_memory=False)
print(f'Dataset shape: {df.shape}')
print(f'\nFirst few rows:')
df.head()

In [ ]:
print('Target variable distribution:')
print(df['readmitted'].value_counts())
print(f'\nPercentage breakdown:')
print(df['readmitted'].value_counts(normalize=True).round(3) * 100)

In [ ]:
# visualize the class distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = ['#2ecc71', '#e74c3c', '#3498db']
df['readmitted'].value_counts().plot(kind='bar', ax=axes[0], color=colors)
axes[0].set_title('Readmission Distribution (Original)', fontsize=13)
axes[0].set_xlabel('Readmission Status')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=0)

missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100)
missing_cols = missing_pct[missing_pct > 0].sort_values(ascending=True)
missing_cols.plot(kind='barh', ax=axes[1], color='#e67e22')
axes[1].set_title('Missing Values by Column (%)', fontsize=13)
axes[1].set_xlabel('Percentage Missing')

plt.tight_layout()
plt.savefig('reports/data_overview.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Data Preprocessing

### 3.1 Handling Missing Values

We drop columns that are mostly empty since they wont help the models. 
Weight is missing about 97% of its values and payer_code about 40%, so both get dropped. 
Medical specialty is about 49% missing but could still be useful, so we fill those with 'Unknown'. 
For race and diagnosis columns, missing values are small enough to fill with the mode.

In [ ]:
print(f'Shape before: {df.shape}')

df = df.drop(columns=['weight', 'payer_code'])
print('Dropped weight and payer_code')

df['medical_specialty'] = df['medical_specialty'].fillna('Unknown')

for col in ['race', 'diag_1', 'diag_2', 'diag_3']:
    if df[col].isnull().any():
        mode_val = df[col].mode()[0]
        count = df[col].isnull().sum()
        print(f'Filled {col} with mode ({mode_val}), {count} values')
        df[col] = df[col].fillna(mode_val)

for col in df.columns:
    if df[col].isnull().any():
        if not pd.api.types.is_numeric_dtype(df[col]):
            df[col] = df[col].fillna(df[col].mode()[0])
        else:
            df[col] = df[col].fillna(df[col].median())

print(f'Remaining missing values: {df.isnull().sum().sum()}')
print(f'Shape after: {df.shape}')

### 3.2 Remove Duplicates and Non-Readmittable Patients

We keep only the first encounter per patient to avoid data leakage between records. 
We also remove patients who died or went to hospice since they obviously cant be readmitted.

In [ ]:
original_len = len(df)

df = df.sort_values('encounter_id')
df = df.drop_duplicates(subset='patient_nbr', keep='first')
print(f'Removed {original_len - len(df)} duplicate patient encounters')

hospice_dead_codes = [11, 13, 14, 19, 20, 21]
mask = df['discharge_disposition_id'].isin(hospice_dead_codes)
print(f'Removed {mask.sum()} expired/hospice records')
df = df[~mask]

df = df.drop(columns=['encounter_id', 'patient_nbr'])
print(f'Final record count: {len(df)}')

### 3.3 Binarize the Target Variable

The original target has three classes: `<30`, `>30`, and `NO`. 
We convert this to a binary problem where 1 means readmitted within 30 days and 0 means not.

In [ ]:
print('Before binarization:')
print(df['readmitted'].value_counts())

df['readmitted'] = (df['readmitted'] == '<30').astype(int)

print('\nAfter binarization:')
print(df['readmitted'].value_counts())
print(f'\nPositive class ratio: {df["readmitted"].mean():.3f}')

### 3.4 Feature Encoding

We need to convert all the categorical features into numbers the models can work with. 
Different columns need different strategies depending on what they represent.

In [ ]:
def map_icd9_codes(code):
    if pd.isna(code):
        return 'Other'
    code = str(code)
    if code.startswith('E'):
        return 'External'
    if code.startswith('V'):
        return 'Supplementary'
    try:
        num = float(code)
    except ValueError:
        return 'Other'
    if 390 <= num <= 459 or num == 785:
        return 'Circulatory'
    elif 460 <= num <= 519 or num == 786:
        return 'Respiratory'
    elif 520 <= num <= 579 or num == 787:
        return 'Digestive'
    elif 250 <= num < 251:
        return 'Diabetes'
    elif 800 <= num <= 999:
        return 'Injury'
    elif 710 <= num <= 739:
        return 'Musculoskeletal'
    elif 580 <= num <= 629 or num == 788:
        return 'Genitourinary'
    elif 140 <= num <= 239:
        return 'Neoplasms'
    elif 240 <= num < 250 or 251 <= num <= 279:
        return 'Endocrine_Other'
    elif 680 <= num <= 709 or num == 782:
        return 'Skin'
    else:
        return 'Other'

for col in ['diag_1', 'diag_2', 'diag_3']:
    df[col] = df[col].apply(map_icd9_codes)
    print(f'{col} categories: {df[col].nunique()}')

In [ ]:
med_mapping = {'No': 0, 'Steady': 1, 'Down': 2, 'Up': 3}
med_columns = [
    'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide',
    'glimepiride', 'acetohexamide', 'glipizide', 'glyburide',
    'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose',
    'miglitol', 'troglitazone', 'tolazamide', 'examide',
    'citoglipton', 'insulin', 'glyburide-metformin',
    'glipizide-metformin', 'glimepiride-pioglitazone',
    'metformin-rosiglitazone', 'metformin-pioglitazone'
]
for col in med_columns:
    if col in df.columns:
        df[col] = df[col].map(med_mapping).fillna(0).astype(int)

age_mapping = {
    '[0-10)': 0, '[10-20)': 1, '[20-30)': 2, '[30-40)': 3,
    '[40-50)': 4, '[50-60)': 5, '[60-70)': 6, '[70-80)': 7,
    '[80-90)': 8, '[90-100)': 9
}
df['age'] = df['age'].map(age_mapping)

invalid_gender = df['gender'].isin(['Unknown/Invalid'])
if invalid_gender.any():
    df = df[~invalid_gender]
    print(f'Removed {invalid_gender.sum()} records with invalid gender')
df['gender'] = df['gender'].map({'Female': 0, 'Male': 1})

df['change'] = df['change'].map({'No': 0, 'Ch': 1})
df['diabetesMed'] = df['diabetesMed'].map({'No': 0, 'Yes': 1})

df['A1Cresult'] = df['A1Cresult'].map({'None': 0, 'Norm': 1, '>7': 2, '>8': 3}).fillna(0).astype(int)
df['max_glu_serum'] = df['max_glu_serum'].map({'None': 0, 'Norm': 1, '>200': 2, '>300': 3}).fillna(0).astype(int)

print('Ordinal encoding complete')

In [ ]:
low_var_dropped = []
for col in med_columns:
    if col in df.columns:
        most_common_pct = df[col].value_counts(normalize=True).iloc[0]
        if most_common_pct > 0.99:
            df = df.drop(columns=[col])
            low_var_dropped.append(col)
print(f'Dropped {len(low_var_dropped)} low-variance medication columns')

top_specialties = df['medical_specialty'].value_counts().nlargest(10).index
df['medical_specialty'] = df['medical_specialty'].apply(
    lambda x: x if x in top_specialties else 'Other'
)

for col in ['admission_type_id', 'discharge_disposition_id', 'admission_source_id']:
    df[col] = df[col].astype(str)

cat_cols = df.select_dtypes(include=['object']).columns.tolist()
if 'readmitted' in cat_cols:
    cat_cols.remove('readmitted')
print(f'One-hot encoding {len(cat_cols)} columns: {cat_cols}')
df = pd.get_dummies(df, columns=cat_cols, drop_first=True)

for col in df.columns:
    if df[col].dtype == bool:
        df[col] = df[col].astype(int)

print(f'Final dataset shape: {df.shape}')

### 3.5 Train/Test Split, Scaling, and SMOTE

We split 80/20 with stratification to maintain the class ratio in both sets. 
Then we standardize features using z-score normalization (fit on training data only to avoid leakage). 
Finally we apply SMOTE to the training set to balance the classes. 
SMOTE is only applied to training data, never to test data.

In [ ]:
X = df.drop(columns=['readmitted'])
y = df['readmitted']
feature_names = list(X.columns)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Training set: {len(X_train)} samples')
print(f'Test set: {len(X_test)} samples')

scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns, index=X_train.index)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns, index=X_test.index)
print('Features scaled')

print(f'\nBefore SMOTE: {dict(pd.Series(y_train).value_counts())}')
smote = SMOTE(random_state=42)
X_train_bal, y_train_bal = smote.fit_resample(X_train_scaled, y_train)
print(f'After SMOTE: {dict(pd.Series(y_train_bal).value_counts())}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

pd.Series(y_train).value_counts().plot(kind='bar', ax=axes[0], color=['#2ecc71', '#e74c3c'])
axes[0].set_title('Before SMOTE', fontsize=13)
axes[0].set_xlabel('Class')
axes[0].set_ylabel('Count')
axes[0].set_xticklabels(['Not Readmitted (0)', 'Readmitted <30 (1)'], rotation=0)

pd.Series(y_train_bal).value_counts().plot(kind='bar', ax=axes[1], color=['#2ecc71', '#e74c3c'])
axes[1].set_title('After SMOTE', fontsize=13)
axes[1].set_xlabel('Class')
axes[1].set_ylabel('Count')
axes[1].set_xticklabels(['Not Readmitted (0)', 'Readmitted <30 (1)'], rotation=0)

plt.tight_layout()
plt.savefig('reports/smote_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Helper Functions

Before training the models, we define a helper function to find the optimal decision threshold. 
By default classifiers use 0.5 as the cutoff, but with imbalanced data that usually means the model 
just predicts the majority class for everything. We sweep thresholds and pick the one that gives the best F1 score.

In [ ]:
def find_best_threshold(y_true, y_prob):
    best_f1 = 0
    best_thresh = 0.5
    for thresh in np.arange(0.1, 0.9, 0.01):
        preds = (y_prob >= thresh).astype(int)
        f1 = f1_score(y_true, preds, zero_division=0)
        if f1 > best_f1:
            best_f1 = f1
            best_thresh = thresh
    return best_thresh, best_f1


def evaluate_model(model, X_test, y_test, model_name):
    start = time.time()
    y_prob = model.predict_proba(X_test)[:, 1]
    inference_time = time.time() - start

    best_thresh, _ = find_best_threshold(y_test, y_prob)
    y_pred = (y_prob >= best_thresh).astype(int)

    metrics = {
        'accuracy': accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred, zero_division=0),
        'recall': recall_score(y_test, y_pred, zero_division=0),
        'f1': f1_score(y_test, y_pred, zero_division=0),
        'roc_auc': roc_auc_score(y_test, y_prob),
        'confusion_matrix': confusion_matrix(y_test, y_pred),
        'y_pred': y_pred,
        'y_prob': y_prob,
        'inference_time': inference_time,
        'threshold': best_thresh,
        'model_name': model_name
    }

    print(f'\n{model_name} Results (threshold={best_thresh:.2f}):')
    print(f'  Accuracy:  {metrics["accuracy"]:.4f}')
    print(f'  Precision: {metrics["precision"]:.4f}')
    print(f'  Recall:    {metrics["recall"]:.4f}')
    print(f'  F1 Score:  {metrics["f1"]:.4f}')
    print(f'  ROC AUC:   {metrics["roc_auc"]:.4f}')
    print(f'  Confusion Matrix:')
    print(f'  {metrics["confusion_matrix"]}')

    return metrics

all_results = {}
print('Helper functions defined, ready to train')

## 5. Model 1: Logistic Regression (Adrian)

Logistic Regression is our baseline model. Its a linear classifier that models the log-odds of the positive class 
as a linear combination of the features. Its the most interpretable of our four models since we can look directly 
at the coefficients to see which features push toward or away from readmission.

We tune the regularization strength (C), penalty type (L1 vs L2), and solver using GridSearchCV with 5-fold cross-validation.

In [ ]:
param_grid_lr = {
    'C': [0.001, 0.01, 0.1, 1, 10],
    'penalty': ['l1', 'l2'],
    'solver': ['liblinear', 'saga'],
    'max_iter': [1000]
}

lr_model = LogisticRegression(random_state=42, class_weight='balanced')

print('Running GridSearchCV for Logistic Regression...')
start = time.time()
lr_search = GridSearchCV(lr_model, param_grid_lr, cv=5, scoring='f1', n_jobs=-1, verbose=0)
lr_search.fit(X_train_bal, y_train_bal)
lr_train_time = time.time() - start

print(f'Best parameters: {lr_search.best_params_}')
print(f'Best CV F1: {lr_search.best_score_:.4f}')
print(f'Training time: {lr_train_time:.2f}s')

In [ ]:
lr_metrics = evaluate_model(lr_search.best_estimator_, X_test_scaled, y_test, 'Logistic Regression')
lr_metrics['train_time'] = lr_train_time
lr_metrics['best_params'] = lr_search.best_params_
all_results['Logistic Regression'] = lr_metrics

In [ ]:
lr_coefs = lr_search.best_estimator_.coef_[0]
lr_importance = pd.DataFrame({
    'feature': feature_names[:len(lr_coefs)],
    'importance': np.abs(lr_coefs),
    'coefficient': lr_coefs
}).sort_values('importance', ascending=False)

print('Top 15 features by absolute coefficient:')
for _, row in lr_importance.head(15).iterrows():
    direction = '+' if row['coefficient'] > 0 else '-'
    print(f"  {direction} {row['feature']}: {row['coefficient']:.4f}")

lr_metrics['importance'] = lr_importance

## 6. Model 2: Random Forest (Niki)

Random Forest is a bagging ensemble that trains multiple decision trees on random subsets of the data and features, 
then aggregates their predictions by majority vote. Each tree sees a different view of the data which helps reduce overfitting.

We use RandomizedSearchCV with 50 iterations since the full parameter grid is too large to search exhaustively.

In [ ]:
param_dist_rf = {
    'n_estimators': [100, 200, 300],
    'max_depth': [10, 20, 30, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2']
}

rf_model = RandomForestClassifier(random_state=42, class_weight='balanced', n_jobs=-1)

print('Running RandomizedSearchCV for Random Forest...')
start = time.time()
rf_search = RandomizedSearchCV(
    rf_model, param_dist_rf, n_iter=50, cv=5,
    scoring='f1', n_jobs=-1, random_state=42, verbose=0
)
rf_search.fit(X_train_bal, y_train_bal)
rf_train_time = time.time() - start

print(f'Best parameters: {rf_search.best_params_}')
print(f'Best CV F1: {rf_search.best_score_:.4f}')
print(f'Training time: {rf_train_time:.2f}s')

In [ ]:
rf_metrics = evaluate_model(rf_search.best_estimator_, X_test_scaled, y_test, 'Random Forest')
rf_metrics['train_time'] = rf_train_time
rf_metrics['best_params'] = rf_search.best_params_
all_results['Random Forest'] = rf_metrics

In [ ]:
rf_importances = rf_search.best_estimator_.feature_importances_
n_feat = min(len(feature_names), len(rf_importances))
rf_importance = pd.DataFrame({
    'feature': feature_names[:n_feat],
    'importance': rf_importances[:n_feat]
}).sort_values('importance', ascending=False)

print('Top 15 features by MDI importance:')
for _, row in rf_importance.head(15).iterrows():
    print(f"  {row['feature']}: {row['importance']:.4f}")

rf_metrics['importance'] = rf_importance

## 7. Model 3: K-Nearest Neighbors (Javi)

KNN is an instance-based learning method that classifies a sample based on the majority class among its K nearest 
neighbors in feature space. Unlike the other models, KNN doesnt learn any parameters during training. Instead it 
stores the training data and does all the work at prediction time by computing distances.

This gives us a fundamentally different approach to compare against our parametric and ensemble models. 
We tune the number of neighbors (K), the distance metric, and the weighting scheme using GridSearchCV.

In [ ]:
param_grid_knn = {
    'n_neighbors': [3, 5, 7, 11, 15, 21],
    'weights': ['uniform', 'distance'],
    'metric': ['euclidean', 'manhattan'],
    'p': [1, 2]
}

knn_model = KNeighborsClassifier(n_jobs=-1)

print('Running GridSearchCV for KNN...')
start = time.time()
knn_search = GridSearchCV(
    knn_model, param_grid_knn, cv=5,
    scoring='f1', n_jobs=-1, verbose=0
)
knn_search.fit(X_train_bal, y_train_bal)
knn_train_time = time.time() - start

print(f'Best parameters: {knn_search.best_params_}')
print(f'Best CV F1: {knn_search.best_score_:.4f}')
print(f'Training time: {knn_train_time:.2f}s')

In [ ]:
knn_metrics = evaluate_model(knn_search.best_estimator_, X_test_scaled, y_test, 'KNN')
knn_metrics['train_time'] = knn_train_time
knn_metrics['best_params'] = knn_search.best_params_
all_results['KNN'] = knn_metrics

# KNN doesnt have built-in feature importances
# so we note that in the comparison
knn_importance = pd.DataFrame({'feature': feature_names, 'importance': [0]*len(feature_names)})
knn_metrics['importance'] = knn_importance
print('Note: KNN has no built-in feature importance method')

## 8. Model 4: XGBoost (Jackson)

XGBoost is a gradient boosting method where each new tree tries to correct the errors of the previous ones. 
Its generally the strongest performer on structured tabular data like this.

We use scale_pos_weight to handle class imbalance and RandomizedSearchCV with 50 iterations for tuning.

In [ ]:
neg_count = (y_train_bal == 0).sum()
pos_count = (y_train_bal == 1).sum()
scale_weight = neg_count / pos_count if pos_count > 0 else 1
print(f'scale_pos_weight: {scale_weight:.2f}')

param_dist_xgb = {
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'max_depth': [3, 5, 7, 10],
    'n_estimators': [100, 200, 300, 500],
    'subsample': [0.6, 0.8, 1.0],
    'colsample_bytree': [0.6, 0.8, 1.0],
    'min_child_weight': [1, 3, 5],
    'gamma': [0, 0.1, 0.2]
}

xgb_model = XGBClassifier(
    random_state=42, scale_pos_weight=scale_weight,
    use_label_encoder=False, eval_metric='logloss', verbosity=0
)

print('Running RandomizedSearchCV for XGBoost...')
start = time.time()
xgb_search = RandomizedSearchCV(
    xgb_model, param_dist_xgb, n_iter=50, cv=5,
    scoring='f1', n_jobs=-1, random_state=42, verbose=0
)
xgb_search.fit(X_train_bal, y_train_bal)
xgb_train_time = time.time() - start

print(f'Best parameters: {xgb_search.best_params_}')
print(f'Best CV F1: {xgb_search.best_score_:.4f}')
print(f'Training time: {xgb_train_time:.2f}s')

In [ ]:
xgb_metrics = evaluate_model(xgb_search.best_estimator_, X_test_scaled, y_test, 'XGBoost')
xgb_metrics['train_time'] = xgb_train_time
xgb_metrics['best_params'] = xgb_search.best_params_
all_results['XGBoost'] = xgb_metrics

In [ ]:
xgb_importances = xgb_search.best_estimator_.feature_importances_
n_feat = min(len(feature_names), len(xgb_importances))
xgb_importance = pd.DataFrame({
    'feature': feature_names[:n_feat],
    'importance': xgb_importances[:n_feat]
}).sort_values('importance', ascending=False)

print('Top 15 features by gain-based importance:')
for _, row in xgb_importance.head(15).iterrows():
    print(f"  {row['feature']}: {row['importance']:.4f}")

xgb_metrics['importance'] = xgb_importance

## 9. Model Comparison and Evaluation

Now we compare all four models side by side across multiple metrics. 
We look at accuracy, precision, recall, F1 score, and ROC AUC. 
For this problem, recall is especially important because missing a patient who will actually be readmitted 
(false negative) is more costly than a false alarm in clinical settings.

### 9.1 Summary Table

In [ ]:
rows = []
for name, metrics in all_results.items():
    rows.append({
        'Model': name,
        'Accuracy': round(metrics['accuracy'], 4),
        'Precision': round(metrics['precision'], 4),
        'Recall': round(metrics['recall'], 4),
        'F1 Score': round(metrics['f1'], 4),
        'ROC AUC': round(metrics['roc_auc'], 4),
        'Threshold': round(metrics['threshold'], 2),
        'Train Time (s)': round(metrics['train_time'], 2)
    })

summary = pd.DataFrame(rows)
print('Model Comparison Summary:')
summary

### 9.2 ROC Curves

In [ ]:
plt.figure(figsize=(10, 8))
for name, metrics in all_results.items():
    fpr, tpr, _ = roc_curve(y_test, metrics['y_prob'])
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, linewidth=2, label=f'{name} (AUC = {roc_auc:.4f})')

plt.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random Classifier')
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('ROC Curves Comparison', fontsize=14)
plt.legend(loc='lower right', fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('reports/roc_curves.png', dpi=150, bbox_inches='tight')
plt.show()

### 9.3 Precision-Recall Curves

In [ ]:
plt.figure(figsize=(10, 8))
for name, metrics in all_results.items():
    precision_vals, recall_vals, _ = precision_recall_curve(y_test, metrics['y_prob'])
    pr_auc = auc(recall_vals, precision_vals)
    plt.plot(recall_vals, precision_vals, linewidth=2, label=f'{name} (AUC = {pr_auc:.4f})')

baseline = y_test.mean()
plt.axhline(y=baseline, color='k', linestyle='--', linewidth=1, label=f'Baseline ({baseline:.3f})')
plt.xlabel('Recall', fontsize=12)
plt.ylabel('Precision', fontsize=12)
plt.title('Precision-Recall Curves', fontsize=14)
plt.legend(loc='upper right', fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('reports/precision_recall_curves.png', dpi=150, bbox_inches='tight')
plt.show()

### 9.4 Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(20, 4))
for ax, (name, metrics) in zip(axes, all_results.items()):
    cm = metrics['confusion_matrix']
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Not Readmit', 'Readmit <30'],
                yticklabels=['Not Readmit', 'Readmit <30'], ax=ax)
    ax.set_title(f'{name}', fontsize=11)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')
plt.tight_layout()
plt.savefig('reports/confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

### 9.5 Metrics Comparison

In [ ]:
metrics_to_plot = ['Accuracy', 'Precision', 'Recall', 'F1 Score', 'ROC AUC']
plot_data = summary.melt(id_vars=['Model'], value_vars=metrics_to_plot, var_name='Metric', value_name='Score')

plt.figure(figsize=(14, 6))
sns.barplot(data=plot_data, x='Metric', y='Score', hue='Model')
plt.title('Model Performance Comparison', fontsize=14)
plt.ylabel('Score')
plt.ylim(0, 1)
plt.legend(title='Model', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig('reports/metrics_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

### 9.6 Feature Importance Comparison

We compare the top features identified by each model. KNN doesnt have built-in feature importances 
so that panel is left blank. If multiple models agree on the same top features, thats a strong signal 
that those features genuinely matter for readmission prediction.

In [ ]:
models_with_importance = {k: v for k, v in all_results.items() if k != 'KNN'}
n_models = len(models_with_importance)
fig, axes = plt.subplots(1, n_models, figsize=(7 * n_models, 8))

if n_models == 1:
    axes = [axes]

for ax, (name, metrics) in zip(axes, models_with_importance.items()):
    imp = metrics['importance'].head(15).copy()
    if len(imp) > 0 and imp['importance'].sum() > 0:
        imp = imp.sort_values('importance', ascending=True)
        ax.barh(imp['feature'], imp['importance'], color='steelblue')
        ax.set_title(f'{name}\nTop 15 Features', fontsize=12)
        ax.set_xlabel('Importance')
    else:
        ax.text(0.5, 0.5, 'No feature\nimportance\navailable',
               ha='center', va='center', transform=ax.transAxes, fontsize=12)
        ax.set_title(f'{name}', fontsize=12)

plt.tight_layout()
plt.savefig('reports/feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

### 9.7 Threshold Analysis

This plot shows how precision, recall, and F1 change as we move the decision threshold for each model. 
The vertical dashed line marks the optimal threshold we selected. This is important because with imbalanced data, 
the default 0.5 threshold almost always results in the model just predicting the majority class.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for ax, (name, metrics) in zip(axes, all_results.items()):
    thresholds = np.arange(0.05, 0.95, 0.01)
    precisions = []
    recalls = []
    f1s = []
    for t in thresholds:
        preds = (metrics['y_prob'] >= t).astype(int)
        precisions.append(precision_score(y_test, preds, zero_division=0))
        recalls.append(recall_score(y_test, preds, zero_division=0))
        f1s.append(f1_score(y_test, preds, zero_division=0))

    ax.plot(thresholds, precisions, label='Precision', linewidth=2)
    ax.plot(thresholds, recalls, label='Recall', linewidth=2)
    ax.plot(thresholds, f1s, label='F1 Score', linewidth=2, linestyle='--')
    ax.axvline(x=metrics['threshold'], color='red', linestyle=':', linewidth=1.5,
              label=f'Best threshold ({metrics["threshold"]:.2f})')
    ax.set_title(f'{name}', fontsize=12)
    ax.set_xlabel('Threshold')
    ax.set_ylabel('Score')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
    ax.set_xlim(0.05, 0.95)
    ax.set_ylim(0, 1)

plt.suptitle('Threshold Analysis by Model', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('reports/threshold_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

### 9.8 Predicted Probability Distributions

These histograms show how each model distributes its predicted probabilities for the two classes. 
A good model should push the positive class (readmitted) to the right and the negative class to the left, 
with minimal overlap. More overlap means the model has a harder time separating the classes.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for ax, (name, metrics) in zip(axes, all_results.items()):
    y_test_arr = np.array(y_test)
    prob_neg = metrics['y_prob'][y_test_arr == 0]
    prob_pos = metrics['y_prob'][y_test_arr == 1]

    ax.hist(prob_neg, bins=50, alpha=0.6, label='Not Readmitted', color='#2ecc71', density=True)
    ax.hist(prob_pos, bins=50, alpha=0.6, label='Readmitted <30', color='#e74c3c', density=True)
    ax.axvline(x=metrics['threshold'], color='black', linestyle='--', linewidth=1.5,
              label=f'Threshold ({metrics["threshold"]:.2f})')
    ax.set_title(f'{name}', fontsize=12)
    ax.set_xlabel('Predicted Probability')
    ax.set_ylabel('Density')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.suptitle('Predicted Probability Distributions', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('reports/probability_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

### 9.9 Calibration Curves

Calibration curves show how well the predicted probabilities match the actual outcomes. 
A perfectly calibrated model would follow the diagonal line, meaning when it says theres a 30% chance 
of readmission, about 30% of those patients actually get readmitted. 
This matters in healthcare because clinicians need to trust the probability estimates, not just the binary predictions.

In [ ]:
plt.figure(figsize=(10, 8))
for name, metrics in all_results.items():
    prob_true, prob_pred = calibration_curve(y_test, metrics['y_prob'], n_bins=10, strategy='uniform')
    plt.plot(prob_pred, prob_true, marker='o', linewidth=2, label=name)

plt.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Perfectly Calibrated')
plt.xlabel('Mean Predicted Probability', fontsize=12)
plt.ylabel('Fraction of Positives', fontsize=12)
plt.title('Calibration Curves', fontsize=14)
plt.legend(loc='lower right', fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('reports/calibration_curves.png', dpi=150, bbox_inches='tight')
plt.show()

### 9.10 Training and Inference Time

In [ ]:
models_list = list(all_results.keys())
train_times = [all_results[m]['train_time'] for m in models_list]
infer_times = [all_results[m]['inference_time'] for m in models_list]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
colors = ['#3498db', '#2ecc71', '#e67e22', '#9b59b6']

ax1.bar(models_list, train_times, color=colors)
ax1.set_title('Training Time', fontsize=13)
ax1.set_ylabel('Time (seconds)')
ax1.tick_params(axis='x', rotation=15)
for i, v in enumerate(train_times):
    ax1.text(i, v + max(train_times)*0.02, f'{v:.1f}s', ha='center', fontsize=10)

ax2.bar(models_list, infer_times, color=colors)
ax2.set_title('Inference Time', fontsize=13)
ax2.set_ylabel('Time (seconds)')
ax2.tick_params(axis='x', rotation=15)
for i, v in enumerate(infer_times):
    ax2.text(i, v + max(infer_times)*0.02, f'{v:.4f}s', ha='center', fontsize=10)

plt.tight_layout()
plt.savefig('reports/runtime_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

### 9.11 Radar Chart Comparison

A radar chart gives us a quick visual summary of how each model performs across all metrics at once. 
The model that covers the most area is generally the best overall performer.

In [ ]:
categories = ['Accuracy', 'Precision', 'Recall', 'F1 Score', 'ROC AUC']
N = len(categories)
angles = [n / float(N) * 2 * np.pi for n in range(N)]
angles += angles[:1]

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
colors = ['#3498db', '#2ecc71', '#e67e22', '#9b59b6']

for i, (name, metrics) in enumerate(all_results.items()):
    values = [metrics['accuracy'], metrics['precision'], metrics['recall'],
              metrics['f1'], metrics['roc_auc']]
    values += values[:1]
    ax.plot(angles, values, 'o-', linewidth=2, label=name, color=colors[i])
    ax.fill(angles, values, alpha=0.1, color=colors[i])

ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories, fontsize=11)
ax.set_ylim(0, 1)
ax.set_title('Model Performance Radar Chart', fontsize=14, pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1), fontsize=11)
ax.grid(True)
plt.tight_layout()
plt.savefig('reports/radar_chart.png', dpi=150, bbox_inches='tight')
plt.show()

### 9.12 Save Results

In [ ]:
os.makedirs('reports', exist_ok=True)
summary.to_csv('reports/model_comparison.csv', index=False)
print('Summary table saved to reports/model_comparison.csv')
print(f'All plots saved to reports/ folder')

best_f1 = max(all_results, key=lambda x: all_results[x]['f1'])
best_recall = max(all_results, key=lambda x: all_results[x]['recall'])
best_auc = max(all_results, key=lambda x: all_results[x]['roc_auc'])

print(f'\nBest model by F1: {best_f1} ({all_results[best_f1]["f1"]:.4f})')
print(f'Best model by Recall: {best_recall} ({all_results[best_recall]["recall"]:.4f})')
print(f'Best model by ROC AUC: {best_auc} ({all_results[best_auc]["roc_auc"]:.4f})')